In [22]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

In [23]:
train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"])
X = train.drop(columns=["hours_spent", "log_hours", "hours_bucket", "project_id"])

def add_features(df):
    df = df.copy()
    df["biggest_commit_share"] = df["max_lines_in_one_commit"] / df["lines_added_total"].replace(0, np.nan)
    df["lines_per_active_day"] = df["lines_of_code"] / df["active_commit_days"].replace(0, np.nan)
    df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"].replace(0, np.nan)
    df["active_day_ratio"] = df["active_commit_days"] / (df["project_span_days"] + 1)
    return df

In [24]:
ORDER_MAPS = {
    "author_prior_projects": {"0": 0, "1-2": 1, "3-5": 2, "6+": 3},
    "author_account_age_years": {"<1":0, "1-2": 1, "3-5": 2, "6+": 3},
}

def encode_ordered(df):
    df = df.copy()
    for col, mapping in ORDER_MAPS.items():
        df[col] = df[col].astype(str).map(mapping)
    return df

In [25]:
def encode_categories(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("category")
    return df

In [26]:
def prepare(df):
    df = add_features(df)
    df = encode_ordered(df)
    df = encode_categories(df)
    return df

X = prepare(X)

C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_12640\3138021966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [27]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(X.shape)

(4500, 73)


In [28]:
from sklearn.metrics import root_mean_squared_log_error

def cv_score(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"],pred)

In [29]:
bool_cols = X.select_dtypes(include="bool").columns
X = X.astype({c: int for c in bool_cols})

cat_cols = X.select_dtypes(include="category").columns.tolist()
X_cat = X.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(str)

In [30]:
import xgboost as xgb
from catboost import CatBoostRegressor

make_lgb = lambda: lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1
)

make_xgb = lambda: xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, enable_categorical=True, tree_method="hist", random_state=42
)

make_cat = lambda: CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=6, cat_features=cat_cols, random_seed=42, verbose=0
)

print("LightGBM: ", round(cv_score(make_lgb, X), 4))
print("XGBoost: ", round(cv_score(make_xgb, X), 4))
print("CatBoost: ", round(cv_score(make_cat, X_cat), 4))

LightGBM:  0.6643


TypeError: Category index must contain only values of the same type, either string or integer. Got values of type `boolean`.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def objective(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 2000),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves = trial.suggest_int("num_leaves", 7, 63),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: lgb.LGBMRegressor(**params, subsample_freq=1, random_state=42, verbose=-1)
    return cv_score(make, X)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best score: ", round(study.best_value, 4))
print("Best settings: ", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]


[W 2026-11-23 20:05:50,860] Trial 0 failed with parameters: {'n_estimators': 1801, 'learning_rate': 0.02869612309584186, 'num_leaves': 61, 'min_child_samples': 64, 'subsample': 0.6563489751446747, 'colsample_bytree': 0.6281619809911464, 'reg_lambda': 0.03236858013134119} because of the following error: NameError("name 'cv_score' is not defined").
Traceback (most recent call last):
  File "d:\Xin Bing\VSC\hour-detective\.venv\Lib\site-packages\optuna\study\_optimize.py", line 210, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_12640\3383863530.py", line 12, in objective
    return cv_score(make, X)
           ^^^^^^^^
NameError: name 'cv_score' is not defined
[W 2026-11-23 20:05:50,866] Trial 0 failed with value None.


NameError: name 'cv_score' is not defined